# Desarrollo de Aplicaciones Web (DAB-2201)
## Sesión 08: Respuestas Semánticas y Excepciones HTTP Personalizadas
**Docente Titular:** MCC. Iván Márquez Larios (`ijmarquezl`)
**Repositorio Maestro:** `https://github.com/ijmarquezl/tecnm-desarrollo-web`

---
### Protocolo de la Sesión
1. **Fase 1 (El Artesano):** Resolver a mano: *Lanzamiento manual de excepciones estructuradas*.
2. **Fase 2 (El Científico):** Definir contrato C-R-E-O: *Manejador global de HTTPException y JSON response* y formular prompt.
3. **Fase 3 (El Auditor):** Someter el servidor al Test Harness: *Pruebas de captura de errores sin crashing (HTTP 500)*.
4. **Entrega:** Documentar en `auditorias/auditoria_sesion08.md`.

### Paso 0: Inicialización del Entorno (FastAPI y PHP 8+)

In [1]:
import os
print('⏳ Instalando FastAPI, Uvicorn y HTTPX para pruebas...')
!pip install -q fastapi uvicorn httpx pydantic passlib python-jose[cryptography] sqlalchemy alembic
if os.system('which php > /dev/null 2>&1') != 0:
    print('⏳ Instalando PHP CLI...')
    !apt-get update -qq && apt-get install -y -qq php-cli > /dev/null 2>&1
print('✅ Entorno Web listo:')
!python3 --version
!php -v | head -n 1

⏳ Instalando FastAPI, Uvicorn y HTTPX para pruebas...
⏳ Instalando PHP CLI...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
✅ Entorno Web listo:
Python 3.13.18
PHP 8.3.6 (cli) (built: Oct 19 2026 15:16:58) (NTS)


---
## SECCIÓN A: TRACK PYTHON / FASTAPI
### Fase 1: El Artesano (15 min - Sin IA)
**Reto:** Lanzamiento manual de excepciones estructuradas

In [2]:
%%writefile app_artesano.py
from fastapi import FastAPI, HTTPException, status
from pydantic import BaseModel

app = FastAPI(title="API Artesano - Sesión 08")

class PayloadRequest(BaseModel):
    dato: str

# --- CÓDIGO DEL PROFE
@app.post("/api/recurso")
def procesar_recurso(payload: PayloadRequest):
    if not payload.dato.strip():
        raise HTTPException(status_code=status.HTTP_400_BAD_REQUEST, detail="Dato vacio")
    return {"status": "success", "mensaje": f"Procesado: {payload.dato}"}

# --- RETO DE MI PARTE ARTEASO
@app.get("/api/error-manual/{codigo}")
def lanzar_error_manual(codigo: int):
    # Aquí validamos a mano y lanzamos la excepción estructurada
    if codigo == 404:
        raise HTTPException(status_code=404, detail="Recurso no encontrado manualmente")
    elif codigo == 400:
        raise HTTPException(status_code=400, detail="Petición incorrecta manual")
    else:
        raise HTTPException(status_code=400, detail="Código de error no soportado")

Writing app_artesano.py


### Fase 2: El Científico (Prompting C-R-E-O)
Pega aquí la implementación optimizada propuesta por la IA:

In [6]:
%%writefile app_ia.py
from fastapi import FastAPI, HTTPException, Request, status
from fastapi.responses import JSONResponse
from pydantic import BaseModel

app = FastAPI(title="API Científico - Sesión 08")

class PayloadRequest(BaseModel):
    dato: str

@app.post("/api/recurso")
def procesar_recurso(payload: PayloadRequest):
    return {"status": "success", "mensaje": f"Procesado: {payload.dato}"}

# --- RETO DEL CIENTÍFICO: Manejador Global de Excepciones ---
# Esto captura CUALQUIER error no controlado (que normalmente daría un 500 feo)
@app.exception_handler(Exception)
async def manejador_global_errores(request: Request, exc: Exception):
    # Retornamos un JSON estructurado en lugar de dejar que el servidor colapse
    return JSONResponse(
        status_code=status.HTTP_500_INTERNAL_SERVER_ERROR,
        content={
            "status": "error",
            "mensaje": "Ha ocurrido un error interno en el servidor.",
            "detalle_tecnico": str(exc), # Útil para debug
            "ruta_afectada": request.url.path
        }
    )

# Ruta para probar el manejador global (provocamos un error a propósito)
@app.get("/api/error-critico")
def provocar_error():
    # Esto lanzará un ZeroDivisionError, que será capturado por el manejador global
    resultado = 10 / 0
    return {"resultado": resultado}

Overwriting app_ia.py


### Fase 3: El Auditor (Test Harness HTTP)
Pruebas de servidor: Pruebas de captura de errores sin crashing (HTTP 500)

In [3]:
import threading
import time
import uvicorn
import httpx
from app_artesano import app

def run_server():
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(1.5)

client = httpx.Client(base_url="http://127.0.0.1:8000")
print("=== AUDITORIA CRITICA WEB: SESION 08 ===\n")
r = client.post("/api/recurso", json={"dato": "Prueba Test Harness"})
print(f"Respuesta Servidor: HTTP {r.status_code} - {r.json()}")
assert r.status_code == 200
print("PASO ✅ - Test Harness HTTP ejecutado con exito.")


=== AUDITORIA CRITICA WEB: SESION 08 ===

Respuesta Servidor: HTTP 200 - {'status': 'success', 'mensaje': 'Procesado: Prueba Test Harness'}
PASO ✅ - Test Harness HTTP ejecutado con exito.


In [1]:
# ESTO ES PARA LA PRUEBA DE LA PARTE DE LA IA QUE FUNCIONE
import os
os.system("fuser -k 8000/tcp > /dev/null 2>&1")
import threading, time, uvicorn, httpx
from app_ia import app

threading.Thread(target=lambda: uvicorn.run(app, host="127.0.0.1", port=8000, log_level="error"), daemon=True).start()
time.sleep(1)

client = httpx.Client(base_url="http://127.0.0.1:8000")
print("Probando el manejador global (división por cero)...")
r = client.get("/api/error-critico")
print(f"Código: {r.status_code}")
print(f"Respuesta JSON controlada: {r.json()}")

Probando el manejador global (división por cero)...
Código: 500
Respuesta JSON controlada: {'status': 'error', 'mensaje': 'Ha ocurrido un error interno en el servidor.', 'detalle_tecnico': 'division by zero', 'ruta_afectada': '/api/error-critico'}
